In [33]:
##导入库，加载数据
##Part B

In [34]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups

RANDOM_SEED = 42

categories = [
    "comp.graphics",
    "sci.space",
    "alt.atheism",
    "talk.religion.misc",
]

# Load the four-category subset and remove metadata and quoted text.
newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_SEED,
)

# Keep document text separate from labels used for later evaluation.
documents = newsgroups.data
true_labels = np.array(newsgroups.target)

print("Number of documents:", len(documents))
print("Number of labels:", len(true_labels))
print("Category names:", newsgroups.target_names)

Number of documents: 3387
Number of labels: 3387
Category names: ['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']


In [35]:
# Build the label mapping from the dataset's actual class order.
label_to_category = dict(enumerate(newsgroups.target_names))

print("Label mapping:")
for label, category in label_to_category.items():
    print(f"{label}: {category}")

# Create a table for inspecting the documents.
text_df = pd.DataFrame({
    "text": documents,
    "label": true_labels,
})

text_df["category"] = text_df["label"].map(label_to_category)

print("\nDocuments per category:")
print(text_df["category"].value_counts().sort_index())

Label mapping:
0: alt.atheism
1: comp.graphics
2: sci.space
3: talk.religion.misc

Documents per category:
category
alt.atheism           799
comp.graphics         973
sci.space             987
talk.religion.misc    628
Name: count, dtype: int64


In [36]:
# Detect documents that contain no text after whitespace removal.
empty_mask = text_df["text"].str.strip().eq("")

print("Empty documents:", int(empty_mask.sum()))

# Inspect the first non-empty document.
example_index = text_df.index[~empty_mask][0]

print("\nExample document index:", example_index)
print("Category:", text_df.loc[example_index, "category"])
print("\nText preview:")
print(text_df.loc[example_index, "text"][:1000])

Empty documents: 92

Example document index: 0
Category: alt.atheism

Text preview:
My point is that you set up your views as the only way to believe.  Saying 
that all eveil in this world is caused by atheism is ridiculous and 
counterproductive to dialogue in this newsgroups.  I see in your posts a 
spirit of condemnation of the atheists in this newsgroup bacause they don'
t believe exactly as you do.  If you're here to try to convert the atheists 
here, you're failing miserably.  Who wants to be in position of constantly 
defending themselves agaist insulting attacks, like you seem to like to do?!
I'm sorry you're so blind that you didn't get the messgae in the quote, 
everyone else has seemed to.


In [37]:
#删去空文档

In [38]:
# Keep only documents containing non-whitespace text.
clean_text_df = text_df.loc[~empty_mask].copy()

# Preserve the original document index for traceability.
clean_text_df = clean_text_df.reset_index(names="original_index")

# Extract aligned documents and evaluation labels.
documents = clean_text_df["text"].tolist()
true_labels = clean_text_df["label"].to_numpy()

print("Original document count:", len(text_df))
print("Removed empty documents:", int(empty_mask.sum()))
print("Remaining document count:", len(documents))
print("Remaining label count:", len(true_labels))

assert len(documents) == len(true_labels)
assert all(document.strip() for document in documents)

print("\nDocuments per category after filtering:")
print(clean_text_df["category"].value_counts().sort_index())

Original document count: 3387
Removed empty documents: 92
Remaining document count: 3295
Remaining label count: 3295

Documents per category after filtering:
category
alt.atheism           779
comp.graphics         955
sci.space             955
talk.religion.misc    606
Name: count, dtype: int64


In [39]:
from sklearn.feature_extraction.text import CountVectorizer

toy_documents = [
    "space science space",
    "computer graphics",
    "space graphics",
]

# Count the occurrences of each vocabulary term in each document.
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(toy_documents)

# Display the small example as a dense table.
bow_table = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=["Document 0", "Document 1", "Document 2"],
)

display(bow_table)

,computer,graphics,science,space
Document 0,0,0,1,2
Document 1,1,1,0,0
Document 2,0,1,0,1


| 概念 | 含义 |
|---|---|
| Stop words（停用词） | 像 `the`、`and` 这样的常见词，实验中可以选择过滤 |
| TF（词频） | 一个词在当前文档中出现的次数；也存在其他计算方式 |
| IDF（逆文档频率） | 衡量一个词在整个文档集合中有多罕见；包含它的文档越少，IDF 通常越大 |

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Calculate TF-IDF weights for the small example.
toy_vectorizer = TfidfVectorizer()
toy_tfidf = toy_vectorizer.fit_transform(toy_documents)

toy_tfidf_table = pd.DataFrame(
    toy_tfidf.toarray(),
    columns=toy_vectorizer.get_feature_names_out(),
    index=["Document 0", "Document 1", "Document 2"],
)

display(toy_tfidf_table.round(3))

# Inspect the inverse document frequency of each term.
idf_table = pd.DataFrame({
    "Term": toy_vectorizer.get_feature_names_out(),
    "IDF": toy_vectorizer.idf_,
})

display(idf_table.round(3))

,computer,graphics,science,space
Document 0,0.000,0.000,0.549,0.836
Document 1,0.796,0.605,0.000,0.000
Document 2,0.000,0.707,0.000,0.707


,Term,IDF
0,computer,1.693
1,graphics,1.288
2,science,1.693
3,space,1.288


In [41]:
# Configure the text representation.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    norm="l2",
)

# Learn the vocabulary and IDF values, then transform the documents.
X_tfidf = tfidf_vectorizer.fit_transform(documents)

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of documents:", X_tfidf.shape[0])
print("Vocabulary size:", len(feature_names))
print("TF-IDF matrix shape:", X_tfidf.shape)
print("Example vocabulary terms:", feature_names[:20])

Number of documents: 3295
Vocabulary size: 17780
TF-IDF matrix shape: (3295, 17780)
Example vocabulary terms: ['00' '000' '0000' '00000' '000005102000' '00041032' '0004422' '0007'
 '000usd' '00196' '0028' '0029' '0038' '0049' '006' '0094' '00index'
 '00pm' '01' '015']


In [42]:
# Measure how many matrix entries are nonzero.
nonzero_count = X_tfidf.nnz
total_entries = X_tfidf.shape[0] * X_tfidf.shape[1]
density = nonzero_count / total_entries

# Find documents with no retained vocabulary terms.
nonzero_per_document = np.asarray(
    X_tfidf.getnnz(axis=1)
).ravel()

zero_vector_mask = nonzero_per_document == 0

print("Nonzero entries:", nonzero_count)
print(f"Matrix density: {density:.4%}")
print("Documents with zero vectors:", int(zero_vector_mask.sum()))

Nonzero entries: 211989
Matrix density: 0.3618%
Documents with zero vectors: 9


In [43]:
# Keep documents with at least one retained vocabulary term.
keep_mask = ~zero_vector_mask

X_tfidf = X_tfidf[keep_mask].tocsr()
clean_text_df = clean_text_df.loc[keep_mask].reset_index(drop=True)

# Keep text and evaluation labels aligned with the matrix rows.
documents = clean_text_df["text"].tolist()
true_labels = clean_text_df["label"].to_numpy()

assert X_tfidf.shape[0] == len(documents) == len(true_labels)
assert np.all(X_tfidf.getnnz(axis=1) > 0)

print("Documents used for clustering:", len(documents))
print("TF-IDF matrix shape:", X_tfidf.shape)
print("Vocabulary size:", len(feature_names))

Documents used for clustering: 3286
TF-IDF matrix shape: (3286, 17780)
Vocabulary size: 17780


In [44]:
def show_document_tfidf(document_index, top_n=10):
    # Get the sparse TF-IDF row for one document.
    row = X_tfidf.getrow(document_index)

    # Sort nonzero entries by descending TF-IDF weight.
    top_positions = np.argsort(row.data)[::-1][:top_n]

    top_terms = pd.DataFrame({
        "Term": feature_names[row.indices[top_positions]],
        "TF-IDF": row.data[top_positions],
    })

    print("Document index:", document_index)
    print(
        "Original index:",
        clean_text_df.loc[document_index, "original_index"],
    )
    print("Category:", clean_text_df.loc[document_index, "category"])
    print("Number of nonzero terms:", row.nnz)

    print("\nText preview:")
    print(documents[document_index][:600])

    print("\nTop TF-IDF terms:")
    display(top_terms.round(4))


# Inspect three example document vectors.
for document_index in [0, 1, 2]:
    show_document_tfidf(document_index)
    print()

Document index: 0
Original index: 0
Category: alt.atheism
Number of nonzero terms: 34

Text preview:
My point is that you set up your views as the only way to believe.  Saying 
that all eveil in this world is caused by atheism is ridiculous and 
counterproductive to dialogue in this newsgroups.  I see in your posts a 
spirit of condemnation of the atheists in this newsgroup bacause they don'
t believe exactly as you do.  If you're here to try to convert the atheists 
here, you're failing miserably.  Who wants to be in position of constantly 
defending themselves agaist insulting attacks, like you seem to like to do?!
I'm sorry you're so blind that you didn't get the messgae in the quote, 
eve

Top TF-IDF terms:


,Term,TF-IDF
0,atheists,0.2920
1,miserably,0.2354
2,condemnation,0.2248
3,failing,0.2248
4,dialogue,0.2206
5,believe,0.2083
6,insulting,0.2079
7,ridiculous,0.2054
8,attacks,0.2054
9,defending,0.2009



Document index: 1
Original index: 1
Category: comp.graphics
Number of nonzero terms: 29

Text preview:

By '8 grey level images' you mean 8 items of 1bit images?
It does work(!), but it doesn't work if you have more than 1bit
in your screen and if the screen intensity is non-linear.

With 2 bit per pixel; there could be 1*c_1 + 4*c_2 timing,
this gives 16 levels, but they are linear if screen intensity is
linear.
With 1*c_1 + 2*c_2 it works, but we have to find the best
compinations -- there's 10 levels, but 16 choises; best 10 must be
chosen. Different compinations for the same level, varies a bit, but
the levels keeps their order.

Readers should verify what I wrote... :-)

Top TF-IDF terms:


,Term,TF-IDF
0,linear,0.4229
1,levels,0.3684
2,screen,0.3333
3,intensity,0.2927
4,16,0.2045
5,level,0.2015
6,images,0.1911
7,10,0.1821
8,best,0.1787
9,bit,0.1635



Document index: 2
Original index: 2
Category: comp.graphics
Number of nonzero terms: 327

Text preview:
FIRST ANNUAL PHIGS USER GROUP CONFERENCE

          The First Annual PHIGS User Group Conference was held March 21-24
          in Orlando, Florida.  The conference was organized by the Rensse-
          laer Design Research Center in co-operation with  IEEE  and  SIG-
          GRAPH.   Attendees  came  from five countries spanning three con-
          tinents.   A  good  cross-section  of  the  PHIGS  community  was
          represented  at this conference with participants including PHIGS
          users, workstation vendors, third-party PHIGS implementors, stan-
          dards  committee

Top TF-IDF terms:


,Term,TF-IDF
0,phigs,0.8708
1,conference,0.1383
2,pex,0.1264
3,tel,0.1239
4,dec,0.1016
5,figaro,0.0849
6,demonstrated,0.0844
7,fax,0.0789
8,extensions,0.0759
9,advanced,0.0729


In [45]:
##Part C

In [46]:
from sklearn.cluster import KMeans

# Configure K-means with four clusters.
kmeans_4 = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=RANDOM_SEED,
)

# Cluster documents using only their TF-IDF vectors.
cluster_labels_4 = kmeans_4.fit_predict(X_tfidf)

print("Number of documents:", len(cluster_labels_4))
print("Number of clusters:", len(np.unique(cluster_labels_4)))
print("Iterations in the selected run:", kmeans_4.n_iter_)
print(f"Inertia: {kmeans_4.inertia_:.4f}")

Number of documents: 3286
Number of clusters: 4
Iterations in the selected run: 33
Inertia: 3212.7604


In [47]:
# Count documents assigned to each cluster.
cluster_sizes = (
    pd.Series(cluster_labels_4, name="Cluster")
    .value_counts()
    .sort_index()
    .rename("Document count")
)

display(cluster_sizes.to_frame())

assert cluster_sizes.sum() == X_tfidf.shape[0]

,Document count
Cluster,
0,1819
1,458
2,319
3,690


In [48]:
# Inspect the cluster assignments of the first ten documents.
assignment_preview = pd.DataFrame({
    "Document index": np.arange(10),
    "Cluster": cluster_labels_4[:10],
    "Text preview": [
        text[:100].replace("\n", " ")
        for text in documents[:10]
    ],
})

display(assignment_preview)

,Document index,Cluster,Text preview
0,0,0,My point is that you set up your views as the ...
1,1,3,By '8 grey level images' you mean 8 items of ...
2,2,3,FIRST ANNUAL PHIGS USER GROUP CONFERENCE ...
3,3,0,"I responded to Jim's other articles today, but..."
4,4,3,"Well, I am placing a file at my ftp today tha..."
5,5,3,I'm also interested in info both public domai...
6,6,0,\tThey did the rollout already??!? I am go...
7,7,0,George William Herbert sez: I like your opti...
8,8,1,"From the ""JPL Universe"" April 23, 1993 VLBI p..."
9,9,2,"]The ""corrupted over and over"" theory is pre..."


In [49]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)

# Compare cluster assignments with true categories after training.
ari_4 = adjusted_rand_score(true_labels, cluster_labels_4)

nmi_4 = normalized_mutual_info_score(
    true_labels,
    cluster_labels_4,
    average_method="arithmetic",
)

# Evaluate cluster separation without using true categories.
silhouette_4 = silhouette_score(
    X_tfidf,
    cluster_labels_4,
    metric="euclidean",
)

print(f"ARI: {ari_4:.4f}")
print(f"NMI: {nmi_4:.4f}")
print(f"Silhouette (Euclidean): {silhouette_4:.4f}")

ARI: 0.2011
NMI: 0.3723
Silhouette (Euclidean): 0.0034


In [50]:
print("Documents per cluster:")
display(cluster_sizes.to_frame())

print(f"Inertia: {kmeans_4.inertia_:.4f}")

# Record the first clustering experiment.
result_4 = pd.DataFrame([{
    "K": 4,
    "ARI": ari_4,
    "NMI": nmi_4,
    "Silhouette (Euclidean)": silhouette_4,
    "Inertia": kmeans_4.inertia_,
}])

display(result_4.round(4))

Documents per cluster:


,Document count
Cluster,
0,1819
1,458
2,319
3,690


Inertia: 3212.7604


,K,ARI,NMI,Silhouette (Euclidean),Inertia
0,4,0.2011,0.3723,0.0034,3212.7604


In [51]:
def get_cluster_top_terms(X, cluster_labels, terms, top_n=10):
    records = []

    for cluster_id in np.unique(cluster_labels):
        # Select all document vectors assigned to this cluster.
        cluster_mask = cluster_labels == cluster_id
        cluster_vectors = X[cluster_mask]

        # Calculate the mean TF-IDF weight of each term.
        mean_tfidf = np.asarray(
            cluster_vectors.mean(axis=0)
        ).ravel()

        # Select the terms with the largest mean weights.
        top_indices = np.argsort(mean_tfidf)[::-1][:top_n]

        for rank, term_index in enumerate(top_indices, start=1):
            records.append({
                "Cluster": int(cluster_id),
                "Document count": int(cluster_mask.sum()),
                "Rank": rank,
                "Term": terms[term_index],
                "Mean TF-IDF": float(mean_tfidf[term_index]),
            })

    return pd.DataFrame(records)


top_terms_4 = get_cluster_top_terms(
    X=X_tfidf,
    cluster_labels=cluster_labels_4,
    terms=feature_names,
    top_n=10,
)

# Display the top terms separately for each cluster.
for cluster_id, cluster_table in top_terms_4.groupby("Cluster"):
    document_count = int(cluster_table["Document count"].iloc[0])

    print(f"Cluster {cluster_id} | Documents: {document_count}")
    display(
        cluster_table[["Rank", "Term", "Mean TF-IDF"]]
        .reset_index(drop=True)
        .round(4)
    )

Cluster 0 | Documents: 1819


,Rank,Term,Mean TF-IDF
0,1,just,0.0162
1,2,don,0.0161
2,3,think,0.0157
3,4,people,0.0131
4,5,like,0.0129
5,6,know,0.0113
6,7,say,0.0102
7,8,does,0.0101
8,9,time,0.0097
9,10,good,0.0092


Cluster 1 | Documents: 458


,Rank,Term,Mean TF-IDF
0,1,space,0.0827
1,2,nasa,0.0364
2,3,launch,0.0269
3,4,shuttle,0.0265
4,5,moon,0.0263
5,6,orbit,0.0257
6,7,mission,0.0210
7,8,like,0.0195
8,9,earth,0.0192
9,10,just,0.0190


Cluster 2 | Documents: 319


,Rank,Term,Mean TF-IDF
0,1,god,0.1385
1,2,jesus,0.0624
2,3,bible,0.0398
3,4,believe,0.0395
4,5,people,0.0343
5,6,christian,0.0272
6,7,say,0.0264
7,8,faith,0.0262
8,9,belief,0.0253
9,10,does,0.0247


Cluster 3 | Documents: 690


,Rank,Term,Mean TF-IDF
0,1,graphics,0.0442
1,2,thanks,0.0421
2,3,image,0.0309
3,4,file,0.0292
4,5,files,0.0279
5,6,format,0.0253
6,7,program,0.0249
7,8,know,0.0247
8,9,ftp,0.0220
9,10,software,0.0219


In [52]:
# Summarize the cluster sizes and top terms in one table.
cluster_summary_4 = (
    top_terms_4.groupby("Cluster", as_index=False)
    .agg(
        Document_count=("Document count", "first"),
        Top_terms=("Term", lambda values: ", ".join(values)),
    )
)

display(cluster_summary_4)

,Cluster,Document_count,Top_terms
0,0,1819,"just, don, think, people, like, know, say, doe..."
1,1,458,"space, nasa, launch, shuttle, moon, orbit, mis..."
2,2,319,"god, jesus, bible, believe, people, christian,..."
3,3,690,"graphics, thanks, image, file, files, format, ..."


In [53]:
# Print all top terms without table truncation.
for row in cluster_summary_4.itertuples(index=False):
    print(f"Cluster {row.Cluster} | Documents: {row.Document_count}")
    print(row.Top_terms)
    print()

Cluster 0 | Documents: 1819
just, don, think, people, like, know, say, does, time, good

Cluster 1 | Documents: 458
space, nasa, launch, shuttle, moon, orbit, mission, like, earth, just

Cluster 2 | Documents: 319
god, jesus, bible, believe, people, christian, say, faith, belief, does

Cluster 3 | Documents: 690
graphics, thanks, image, file, files, format, program, know, ftp, software



In [54]:
# Compare learned clusters with true categories after training.
cluster_category_counts = pd.crosstab(
    pd.Series(cluster_labels_4, name="Cluster"),
    pd.Series(
        clean_text_df["category"].to_numpy(),
        name="True category",
    ),
)

display(cluster_category_counts)

# Show the category proportions within each cluster.
cluster_category_percentages = (
    cluster_category_counts
    .div(cluster_category_counts.sum(axis=1), axis=0)
    .mul(100)
)

print("Category percentages within each cluster:")
display(cluster_category_percentages.round(1))

True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,592,299,471,457
1,5,7,445,1
2,175,0,0,144
3,6,644,38,2


Category percentages within each cluster:


True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,32.5,16.4,25.9,25.1
1,1.1,1.5,97.2,0.2
2,54.9,0.0,0.0,45.1
3,0.9,93.3,5.5,0.3


In [55]:
# Print complete term lists without truncation.
for row in cluster_summary_4.itertuples(index=False):
    print(f"Cluster {row.Cluster} | Documents: {row.Document_count}")
    print(row.Top_terms)
    print()

Cluster 0 | Documents: 1819
just, don, think, people, like, know, say, does, time, good

Cluster 1 | Documents: 458
space, nasa, launch, shuttle, moon, orbit, mission, like, earth, just

Cluster 2 | Documents: 319
god, jesus, bible, believe, people, christian, say, faith, belief, does

Cluster 3 | Documents: 690
graphics, thanks, image, file, files, format, program, know, ftp, software



In [56]:
# Use true categories only to inspect the completed clustering.
cluster_category_counts = pd.crosstab(
    pd.Series(cluster_labels_4, name="Cluster"),
    pd.Series(
        [label_to_category[label] for label in true_labels],
        name="True category",
    ),
)

display(cluster_category_counts)

# Calculate category percentages within each cluster.
cluster_category_percentages = (
    cluster_category_counts
    .div(cluster_category_counts.sum(axis=1), axis=0)
    * 100
)

print("Category percentages within each cluster:")
display(cluster_category_percentages.round(1))

True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,592,299,471,457
1,5,7,445,1
2,175,0,0,144
3,6,644,38,2


Category percentages within each cluster:


True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,32.5,16.4,25.9,25.1
1,1.1,1.5,97.2,0.2
2,54.9,0.0,0.0,45.1
3,0.9,93.3,5.5,0.3


In [57]:
# Reuse the completed K=4 experiment.
kmeans_models = {4: kmeans_4}
cluster_assignments = {4: cluster_labels_4}

clustering_results = [{
    "K": 4,
    "ARI": ari_4,
    "NMI": nmi_4,
    "Silhouette (Euclidean)": silhouette_4,
    "Inertia": kmeans_4.inertia_,
    "Random seed": RANDOM_SEED,
}]

# Change only the number of clusters.
for k in [2, 6, 8]:
    print(f"Running K={k}...")

    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=RANDOM_SEED,
    )

    # Train without using true category labels.
    cluster_labels = model.fit_predict(X_tfidf)

    # Use true labels only for external evaluation.
    ari = adjusted_rand_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(
        true_labels,
        cluster_labels,
        average_method="arithmetic",
    )

    # Calculate the full-data silhouette score.
    silhouette = silhouette_score(
        X_tfidf,
        cluster_labels,
        metric="euclidean",
    )

    kmeans_models[k] = model
    cluster_assignments[k] = cluster_labels

    clustering_results.append({
        "K": k,
        "ARI": ari,
        "NMI": nmi,
        "Silhouette (Euclidean)": silhouette,
        "Inertia": model.inertia_,
        "Random seed": RANDOM_SEED,
    })

    print(f"Completed K={k}")

results_df = (
    pd.DataFrame(clustering_results)
    .sort_values("K")
    .reset_index(drop=True)
)

display(results_df.round(4))

Running K=2...
Completed K=2
Running K=6...
Completed K=6
Running K=8...
Completed K=8


,K,ARI,NMI,Silhouette (Euclidean),Inertia,Random seed
0,2,0.1942,0.3168,0.0039,3233.9953,42
1,4,0.2011,0.3723,0.0034,3212.7604,42
2,6,0.1539,0.3145,0.0027,3199.0620,42
3,8,0.1213,0.3042,0.0039,3188.5035,42


In [58]:
# Inspect cluster sizes for each experiment.
for k in [2, 4, 6, 8]:
    sizes = (
        pd.Series(cluster_assignments[k], name="Cluster")
        .value_counts()
        .sort_index()
        .rename("Document count")
    )

    print(f"K={k}")
    display(sizes.to_frame())

K=2


,Document count
Cluster,
0,834
1,2452


K=4


,Document count
Cluster,
0,1819
1,458
2,319
3,690


K=6


,Document count
Cluster,
0,240
1,464
2,1460
3,307
4,191
5,624


K=8


,Document count
Cluster,
0,15
1,1486
2,416
3,311
4,234
5,375
6,195
7,254


In [59]:
# Inspect top terms using the existing cluster assignments.
for k in [2, 6, 8]:
    top_terms = get_cluster_top_terms(
        X=X_tfidf,
        cluster_labels=cluster_assignments[k],
        terms=feature_names,
        top_n=10,
    )

    print(f"K={k}")

    for cluster_id, table in top_terms.groupby("Cluster"):
        document_count = int(table["Document count"].iloc[0])
        terms = ", ".join(table["Term"])

        print(f"Cluster {cluster_id} | Documents: {document_count}")
        print(f"Top terms: {terms}")
        print()

K=2
Cluster 0 | Documents: 834
Top terms: graphics, thanks, image, file, files, program, know, format, ftp, software

Cluster 1 | Documents: 2452
Top terms: god, just, don, think, people, space, like, know, say, does

K=6
Cluster 0 | Documents: 240
Top terms: file, image, files, format, gif, tiff, program, images, convert, formats

Cluster 1 | Documents: 464
Top terms: graphics, thanks, looking, know, 3d, help, does, hi, mail, advance

Cluster 2 | Documents: 1460
Top terms: just, like, know, time, does, don, did, kent, cheers, ve

Cluster 3 | Documents: 307
Top terms: space, nasa, shuttle, launch, orbit, mission, hst, moon, satellite, earth

Cluster 4 | Documents: 191
Top terms: god, jesus, believe, bible, faith, belief, does, christian, say, christians

Cluster 5 | Documents: 624
Top terms: people, think, don, just, say, objective, like, moral, morality, right

K=8
Cluster 0 | Documents: 15
Top terms: wk, 9591, mcwilliams, 517, 2178, 18084tm, 336, msu, hm, cl

Cluster 1 | Documents: 1

In [60]:
# Calculate top terms using the existing K=8 assignments.
top_terms_8 = get_cluster_top_terms(
    X=X_tfidf,
    cluster_labels=cluster_assignments[8],
    terms=feature_names,
    top_n=10,
)

# Display one compact line per cluster.
for cluster_id, table in top_terms_8.groupby("Cluster"):
    document_count = int(table["Document count"].iloc[0])
    terms = ", ".join(table["Term"])

    print(f"Cluster {cluster_id} ({document_count} documents): {terms}")

Cluster 0 (15 documents): wk, 9591, mcwilliams, 517, 2178, 18084tm, 336, msu, hm, cl
Cluster 1 (1486 documents): just, like, don, know, does, time, did, good, say, ve
Cluster 2 (416 documents): space, nasa, launch, shuttle, moon, orbit, mission, earth, just, like
Cluster 3 (311 documents): graphics, software, edu, 3d, computer, comp, ftp, available, pc, package
Cluster 4 (234 documents): thanks, looking, advance, hi, know, program, help, mail, ftp, appreciated
Cluster 5 (375 documents): think, people, don, objective, moral, morality, just, right, like, say
Cluster 6 (195 documents): file, image, files, format, gif, bit, tiff, images, convert, program
Cluster 7 (254 documents): god, jesus, bible, believe, faith, said, christians, christian, belief, say


In [61]:
# Inspect documents in the small cluster from the K=8 experiment.
small_cluster_indices = np.flatnonzero(
    cluster_assignments[8] == 0
)

print("Number of documents:", len(small_cluster_indices))

# Show the beginning and ending of three example documents.
for document_index in small_cluster_indices[:3]:
    text = documents[document_index]

    print(f"\nDocument index: {document_index}")
    print(f"Character count: {len(text)}")

    if len(text) <= 1600:
        print("\nFull text:")
        print(text)
    else:
        print("\nBeginning:")
        print(text[:800])

        print("\nEnding:")
        print(text[-800:])

Number of documents: 15

Document index: 7
Character count: 531

Full text:
George William Herbert sez:


I like your optimism, George.  I don't know doots about raising that kind
of dough, but if you need people to split the work and the $700M, you just
give me a ring :-)  Living alone for a year on the moon sounds horrid, but
I'd even try that, if I got a bigger cut.  :-)

-Tommy Mac
-------------------------------------------------------------------------
Tom McWilliams 517-355-2178 wk   \\ As the radius of vision increases,
18084tm@ibm.cl.msu.edu 336-9591 hm \\ the circumference of mystery grows.

Document index: 19
Character count: 374

Full text:
Pat sez;

Yeah, but a windscreen cut down most of it.  Canopies ended it completely.

Of course, the environment in space continues to suck :-)

-Tommy Mac
-------------------------------------------------------------------------
Tom McWilliams 517-355-2178 wk   \\ As the radius of vision increases,
18084tm@ibm.cl.msu.edu 336-9591 hm \\ 

In [62]:
# Record the settings and results of all clustering experiments.
experiment_log = results_df.copy()

experiment_log.insert(
    0,
    "Experiment",
    experiment_log["K"].map(
        lambda k: f"tfidf_kmeans_k{k}"
    ),
)

experiment_log["Model"] = "K-means"
experiment_log["Input representation"] = "TF-IDF"
experiment_log["Number of documents"] = X_tfidf.shape[0]
experiment_log["Number of features"] = X_tfidf.shape[1]

experiment_log["Vectorizer settings"] = (
    "lowercase=True, stop_words=english, "
    "min_df=2, max_df=0.95, norm=l2"
)

experiment_log["K-means settings"] = (
    "init=k-means++, n_init=10, max_iter=300"
)

# Inertia is the final K-means training objective.
experiment_log["Training objective"] = (
    "Sum of squared distances to assigned cluster centers; see Inertia"
)

observations = {
    2: (
        "Graphics-related cluster and a large mixed cluster."
    ),
    4: (
        "Highest ARI and NMI among tested K values; "
        "clear space and graphics clusters, with substantial mixing."
    ),
    6: (
        "Finer themes include image formats, graphics help, "
        "space, religion, and ethics; a large mixed cluster remains."
    ),
    8: (
        "Finer themes and a 15-document cluster "
        "apparently influenced by repeated residual signatures."
    ),
}

experiment_log["Observations"] = experiment_log["K"].map(observations)

display(experiment_log)

,Experiment,K,ARI,NMI,Silhouette (Euclidean),Inertia,Random seed,Model,Input representation,Number of documents,Number of features,Vectorizer settings,K-means settings,Training objective,Observations
0,tfidf_kmeans_k2,2,0.194215,0.316812,0.003934,3233.995302,42,K-means,TF-IDF,3286,17780,"lowercase=True, stop_words=english, min_df=2, ...","init=k-means++, n_init=10, max_iter=300",Sum of squared distances to assigned cluster c...,Graphics-related cluster and a large mixed clu...
1,tfidf_kmeans_k4,4,0.201077,0.372321,0.003444,3212.760370,42,K-means,TF-IDF,3286,17780,"lowercase=True, stop_words=english, min_df=2, ...","init=k-means++, n_init=10, max_iter=300",Sum of squared distances to assigned cluster c...,Highest ARI and NMI among tested K values; cle...
2,tfidf_kmeans_k6,6,0.153860,0.314530,0.002682,3199.061953,42,K-means,TF-IDF,3286,17780,"lowercase=True, stop_words=english, min_df=2, ...","init=k-means++, n_init=10, max_iter=300",Sum of squared distances to assigned cluster c...,"Finer themes include image formats, graphics h..."
3,tfidf_kmeans_k8,8,0.121264,0.304244,0.003912,3188.503523,42,K-means,TF-IDF,3286,17780,"lowercase=True, stop_words=english, min_df=2, ...","init=k-means++, n_init=10, max_iter=300",Sum of squared distances to assigned cluster c...,Finer themes and a 15-document cluster apparen...


# Week 2 概念理解与实验总结

## 1. Bag-of-Words

Bag-of-Words（词袋）通过统计词语出现的次数，将文本转换为数值向量。
首先建立词表，每个词对应一个固定的向量位置，然后记录每篇文档中
各个词出现的次数。例如，“space science space”中 space 出现两次，
science 出现一次。词袋表示容易理解，但忽略了词序，也不能直接理解
词义。两句话即使表达的意思不同，只要词语及其出现次数相同，就会得到相同的词袋向量。

## 2. TF-IDF

TF-IDF 在词频的基础上考虑词语在整个文档集合中的常见程度。
TF 描述词语在当前文档中的出现情况，IDF 则降低普遍出现在许多
文档中的词的权重。一个词在当前文档中频繁出现、在其他文档中相对少见时，通常会得到较高的 TF-IDF 权重。

本实验使用 TfidfVectorizer，并过滤英文停用词和只出现在一篇
文档中的词。最终每篇文档由一个 17780 维的向量表示。由于单篇
文档只包含词表中的少量词语，矩阵大部分位置为零，因此使用稀疏格式存储。TF-IDF 反映统计特征，并不意味着模型理解了词语的语义。

## 3. K-means

K-means 将样本分成预先指定的 K 个簇。它反复将样本分配给最近的
簇中心，再用簇内样本向量的平均值更新中心。它优化的目标是所有样本到所属簇中心的平方距离之和，即 inertia。

本实验只用 TF-IDF 向量训练 K-means，没有把真实类别标签作为输入。
K 是超参数，簇编号没有固定的类别含义，各簇大小也不必相等。增加 K 可以降低簇内距离，但不保证聚类结果更符合真实类别。

## 4. ARI

ARI 用于比较聚类分组与真实类别的一致程度。它考察样本对是否被
两个分组同时放在一起或分开，并对随机一致性进行校正。完全一致时
ARI 为 1，随机分组的期望值约为 0，也可能出现负值。它不要求簇编号
与类别编号相同，也不是分类准确率。本次实验中，K=4 的 ARI 最高，为 0.2011，表明其分组与真实类别有一定一致性，但仍有明显差距。

## 5. NMI

NMI 衡量聚类分组与真实类别共享的信息，并将结果归一化到 0 到 1。
数值越高，说明两个分组的关联越强。它同样不受簇编号重命名的影响，
但与 ARI 不同，NMI 没有进行相同的随机一致性校正。本实验中，K=4 的 NMI 最高，为 0.3723。ARI 和 NMI 都只在聚类完成后用于评价。

## 6. 实验观察

原始数据包含 3387 篇文档。移除 92 篇空文档以及 9 篇 TF-IDF
零向量文档后，最终对 3286 篇文档进行聚类。IDF 在移除零向量前的 3295 篇非空文档上拟合，之后没有重新拟合。

在 K=2、4、6、8 中，K=4 的 ARI 和 NMI 最高，但所有 Silhouette
分数都接近零，说明当前表示和欧氏距离下的簇间分离较弱。
更大的 K 揭示了图像格式、软件资源和伦理等细分内容，但没有提高与原始类别的一致性。

抽查 K=8 的一个小簇时，发现部分文档含有相同的残留签名，
其姓名、号码和邮箱片段也出现在该簇的代表词中。这说明聚类可能
受到重复格式信息影响。后续可以改进签名清理、比较多个随机种子，并尝试其他文本表示。